# Exercise 1. Fine-Tuning for Multi-Label Classification 


## 1.1 Setup: Load Packages & Data
If you have not already, please download `transformers`, `evaluate`, `torch`, `numpy` and `datasets` (in `venv` or in UCloud) in your terminal:
```bash
pip install transformers torch numpy evaluate datasets
```

:::{admonition} Download in Jupyter
:class: tip
Remember, you can also download the packages in Jupyter with the `%pip` magic command as we have done in previous classes.
:::

In [1]:
from pathlib import Path
from datasets import load_dataset, ClassLabel
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
import torch
import evaluate
import numpy as np

### Data

In [2]:
path = Path.cwd()
data_path = path.parents[1] / "resources" / "data" / "hf" # path for huggingface datasets (so we don't have to redownload them every time)

In [3]:
ds = load_dataset("SetFit/student-question-categories", split="train", cache_dir=data_path)

Repo card metadata block was not found. Setting CardData to empty.


Print the dataset + an example of a text:

In [4]:
print(ds)

Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 117519
})


In [5]:
# let's print an example
print(ds["text"][12])

Hydroponic is a subset of what type of culture?
A. Hydroculture
B. Solid medium culture
c. xeroculture
D. Tissue culture


#### Label Column
Let's define the label column:

In [6]:
num_classes = 4
ds = ds.cast_column("label", ClassLabel(num_classes=num_classes))

We'll split into train and val and downsample to 2000 train examples and 500 test examples to make it run faster for today's class:

In [7]:
# split into train/val
ds = ds.train_test_split(train_size=2000,test_size=500, seed=42, stratify_by_column="label")
train_data = ds["train"]
val_data = ds["test"]

## 1.2 Loading the Model
We'll load the BERT model `distilbert-base-cased` and its corresponding tokenizer. The suffix `cased` tells us that this `DistilBERT` is sensitive to letter case (distinguishing between `english` and `English`).

`DistilBERT` also exists in [uncased](https://huggingface.co/distilbert/distilbert-base-uncased) and [multilingual](https://huggingface.co/distilbert/distilbert-base-multilingual-cased) versions. 

In [8]:
# define model + where to load it from (if already downloaded/cached)
model_path = path.parents[1] / "resources" / "models" / "hf"
model_id = "distilbert/distilbert-base-cased"

# GPU or CPU? Default to CPU if no GPU available


# load model + tokenizer
model = AutoModelForSequenceClassification.from_pretrained(
                                                            model_id, 
                                                            num_labels=num_classes, # pre-defined number of labels in our dataset
                                                            cache_dir=model_path, 
                                                           ) 
tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=model_path)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


:::{admonition} "Some weights of DistilBertForSequenceClassification ..." ? 
:class: tip, dropdown
The message above means we *have* to fine-tune the model, since its classification head (weights) is newly initialized. On huggingface.co, you can also find BERT models that have already been fine-tuned for classification.
:::

Let's look more into our model by printing its parameters:

In [9]:
print(model.num_parameters())

65784580


:::{admonition} QUESTION
:class: red
`DistilBERT` has 65.M parameters. From what you might have heard about `Large Language Models` - do you know where this would range? Is this a lot?
:::

## 1.3 Tokenization
We can use `DistilBERT`'s trained tokenizer to represent the text in our dataset:

In [10]:
def preprocess_function(examples):
   """Tokenize input data"""
   return tokenizer(examples["text"], truncation=True)

:::{admonition} QUESTION
:class: red
Can you identify a way to make the function above better in terms of how it is defined and how it is described? Is it easily applicable to other datasets? Why/Why not?

<details>
  <summary>ANSWER</summary>
  I would consider to ...
  <ol>
    <li>Rename the function to <code>tokenize</code>, making its name more informative to its purpose.</li>
    <li>Add more details in the docstring (and <a href="https://docs.python.org/3/library/typing.html">type hints</a>) about the expected input and output, instead of only writing <code>"""Tokenize input data"""</code>.</li>
    <li>Make the function more generalizable by adding a <code>text_col</code> parameter, allowing the user to specify a different column name (e.g., our text column being called <code>"generation"</code>)</li>
  </ol>
</details>
:::

We use the `.map` method to use the tokenize function on each row in our dataset!

In [11]:
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_val = val_data.map(preprocess_function, batched=True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

## 1.4 Training Arguments and Parameters!
To `fine-tune` our BERT model, we'll use the `Trainer` class. This needs quite a lot of information. We'll break this down in this section!

```python
trainer = Trainer(
   model=model,
   args=training_args,              # how many examples per batch? how many times?
   tokenizer=tokenizer,             
   data_collator=data_collator,     # ensure equal length with Padding
   compute_metrics=compute_metrics, # evaluation function
   train_dataset=tokenized_train,   # data
   eval_dataset=tokenized_val,      # data
)
```

### Training Arguments

Training arguments let us control how the model learns. Key examples include:

<div style="display: inline-block; text-align: left; margin-left: 40px; line-height: 1.5;">
  <div><code style="display: inline-block; width: 120px;">learning_rate</code> -> how fast the model learns</div>
  <div><code style="display: inline-block; width: 120px;">X_batch_size</code> -> how many examples it sees at once</div>
  <div><code style="display: inline-block; width: 120px;">epochs</code> -> how many times it goes through all batches</div>
</div>
<br><br>
We define these arguments in code like this:

In [ ]:
batch_size = 8
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=batch_size,
   per_device_eval_batch_size=batch_size,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none"
)

### Padding!
BERT requires all input sequences to be the same length, but text data naturally has different lengths.   

To handle this, we `pad` shorter sequences with zeroes to match the longest one in the batch. An **attention mask** then tells BERT to ignore the padded parts, so it only focuses on the actual text. A visualisation of this is:

```{figure} ../figures/class5/padding.png
---
name: padding-illustration
width: 100%
---
Figure from {cite:t}`tunstall2022nlp`, chapter 2.
```

In code, this can be expressed as a `data_collator`:

In [18]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### Evaluation

In [13]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    f1_metric = evaluate.load("f1")
    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"  # or "weighted"
    )["f1"]
    return {"f1": f1}

### Train
We are now ready to train :)

In [15]:
# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_val,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)

/var/folders/gg/gk923hkx2w3bw72pk2shplydry9j0b/T/ipykernel_81263/1138016704.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [16]:
trainer.train()

/Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


TrainOutput(global_step=250, training_loss=0.586315673828125, metrics={'train_runtime': 46.7773, 'train_samples_per_second': 42.756, 'train_steps_per_second': 5.344, 'total_flos': 107209275485760.0, 'train_loss': 0.586315673828125, 'epoch': 1.0})

In [17]:
trainer.evaluate()

/Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.41761693358421326,
 'eval_f1': 0.8598992300991953,
 'eval_runtime': 5.0365,
 'eval_samples_per_second': 99.275,
 'eval_steps_per_second': 12.509,
 'epoch': 1.0}

### Your Turn: Re-create the Pipeline in a Python Script
:::{admonition} HANDS-ON
:class: red
This task focuses on practicing how to create pipelines in scripts. You should do the following: 
1. Draw the DistilBERT fine-tuning pipeline as a diagram on a piece of paper (or digitally in powerpoint). What are the different steps when fine-tuning?
2. Take the snippets above and write them in a Python script
3. Run the script!


**You may structure the script however you like!** You can choose to keep most snippets inside of `main()` or define additional helper functions outside of `main()`, calling them inside of `main()`. See also [Python Scripts](../python_scripts.md).   

For the more advanced coder, try to make the script as generalizable as possible to other BERT models or datasets (e.g., through [argparse](https://docs.python.org/3/howto/argparse.html#introducing-optional-arguments)).
:::